# 03 — LoRA training on Google Colab

Trains the **Ravi Varma style LoRA** (Stable Diffusion 1.5 + PEFT) end to end:
dataset check → smoke test → full training → checkpoint inspection → generation → export.

**Runtime:** Runtime → Change runtime type → **T4 GPU**, and **Runtime version → 2026.07**.
The newest Colab image moved to Python 3.13 (2026-09-16), which the pinned stack
(transformers 4.44 / tokenizers 0.19 / pillow 10.4) has no wheels for; 2026.07 is the last
Python 3.12 image and was used for the verified run (torch 2.11.0+cu128, Tesla T4).
Full training (1500 optimizer steps × 4 accumulation) takes roughly 1–1.5 h on a T4.

**Before you start**, on your own machine (where `data/raw/` lives):

```bash
python scripts/validate_dataset.py
python scripts/generate_captions.py      # BLIP + BLIP-VQA template captions
python scripts/prepare_dataset.py        # aspect buckets + held-out reference set
python scripts/package_dataset.py        # -> dist/ravi_varma_dataset.zip (~30 MB)
```

and upload `dist/ravi_varma_dataset.zip` to Google Drive (or upload it in step 4).
Option B in step 4 rebuilds the dataset from scratch on Colab instead.

> Generated images are AI-generated, Ravi Varma–*inspired* artwork — not authentic paintings by Raja Ravi Varma.

## 1. GPU verification

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "|", torch.cuda.get_device_name(0))

## 2. Project setup and dependencies

Clones the repo (skip if you uploaded it) and installs the pinned training stack.
Colab's own torch/torchvision are kept — see the comments in `requirements-colab.txt`.

In [ ]:
import os
REPO_URL = "https://github.com/f20240684-cmd/the-next-ravi-varma.git"
PROJECT = "/content/the-next-ravi-varma"

if not os.path.exists(PROJECT):
    !git clone -q {REPO_URL} {PROJECT}
%cd {PROJECT}
!git log --oneline -1
!pip install -q -r requirements-colab.txt
!pip install -q -e . --no-deps

## 3. torchao compatibility check

Some Colab images ship a `torchao` that does not match the installed torch, which breaks
`import diffusers`/`peft`. This cell only acts if that import actually fails: first it applies
the pin that worked for this project before, and if that is not enough it removes `torchao`
(this project does not use it).

In [ ]:
import subprocess, sys

def ml_stack_ok():
    probe = "import diffusers, peft, accelerate; from diffusers import StableDiffusionPipeline; from peft import LoraConfig"
    r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
    return r.returncode == 0, r.stderr[-2000:]

ok, err = ml_stack_ok()
if not ok and "torchao" in err:
    print("torchao incompatibility detected -> pinning torchao==0.18.0")
    !pip install -q --upgrade --no-deps "torchao==0.18.0"
    ok, err = ml_stack_ok()
    if not ok and "torchao" in err:
        print("still failing -> uninstalling torchao (unused by this project)")
        !pip uninstall -y -q torchao
        ok, err = ml_stack_ok()
print("ML stack OK" if ok else err)
assert ok, "Fix the import error above before training."
!python -c "import torch, diffusers, peft, transformers, accelerate; print('torch', torch.__version__, '| diffusers', diffusers.__version__, '| peft', peft.__version__, '| transformers', transformers.__version__, '| accelerate', accelerate.__version__)"

## 4. Load the dataset

**Option A (default)** — unzip the prepared dataset made by `scripts/package_dataset.py`.
Point `DATASET_ZIP` at your Drive copy, or leave it as `None` to upload the file now.

In [ ]:
DATASET_ZIP = "/content/drive/MyDrive/ravi_varma/ravi_varma_dataset.zip"   # or None to upload

if DATASET_ZIP and DATASET_ZIP.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
if not DATASET_ZIP or not os.path.exists(DATASET_ZIP):
    from google.colab import files
    DATASET_ZIP = next(iter(files.upload()))
!unzip -q -o "{DATASET_ZIP}" -d {PROJECT}
!ls data/processed | wc -l; ls data/reference | wc -l

**Option B (full rebuild, ~1 h)** — re-download the exact raw images listed in the committed
`data/metadata/metadata.jsonl`, then validate, caption and prepare here. Only run this if you did
not use Option A.

In [ ]:
REBUILD_FROM_RAW = False

if REBUILD_FROM_RAW:
    !python scripts/download_dataset.py --from-metadata      # Wikimedia rate limits make this slow
    !python scripts/validate_dataset.py
    !pip install -q open-clip-torch==2.26.1
    !python scripts/generate_captions.py --device cuda
    !python scripts/prepare_dataset.py

## 5. Dataset validation

Checks the manifest the trainer will read: every image exists and is a valid bucket size, every
caption starts with `<rvvarma>`, fits CLIP's 77 tokens and is not placeholder text.

In [ ]:
!python scripts/validate_dataset.py --manifest

import json, collections
rows = [json.loads(l) for l in open("data/processed/manifest.jsonl")]
print(len(rows), "training samples | resolutions:", collections.Counter(f"{r['width']}x{r['height']}" for r in rows).most_common())
for r in rows[:3]:
    print("-", r["image"], "|", r["text"])

## 6. Caption inspection

Spot-check a few structured captions (all VQA questions and answers are stored per image).

In [ ]:
import json, random
from IPython.display import display
from PIL import Image

random.seed(0)
for r in random.sample(rows, 3):
    stem = os.path.splitext(os.path.basename(r["image"]))[0]
    cap = json.load(open(f"data/captions/{stem}.json"))
    im = Image.open(r["image"]); im.thumbnail((256, 256)); display(im)
    print(r["title"], "\n  caption:", cap["flat_caption"], "\n  VQA:", cap["qa"], "\n")

## 7. Smoke test (≈3 min)

Twenty optimizer steps into a throw-away directory, with a checkpoint and validation images at
step 10. Proves the data loader, fp16 training, checkpoint/LoRA export and validation all work
before committing an hour of GPU time.

In [ ]:
!python scripts/train_lora.py --config configs/lora_sd15.yaml --smoke-test
!ls -R outputs/smoke_test/lora | head -20

## 8. Full training

Uses `configs/lora_sd15.yaml` unchanged: rank 16, alpha 16, dropout 0.05, `to_q/to_k/to_v/to_out.0`,
LR 1e-4 (constant, 100 warm-up steps), batch 1 × grad-accum 4, 1500 steps, fp16, seed 42.
A checkpoint (with loadable LoRA weights) and validation images are written every 250 steps.

If the runtime disconnects, just re-run this cell: it resumes from the latest checkpoint.
Optionally mount Drive and set `checkpointing.output_dir` to a Drive path so checkpoints survive.

In [ ]:
!python scripts/train_lora.py --config configs/lora_sd15.yaml

## 9. Checkpoint inspection

Loss curve, run summary and the validation grids at every checkpoint (step 0 = untrained baseline).

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, glob
loss = pd.read_csv("checkpoints/lora/ravi_varma/loss_history.csv")
ax = loss.plot(x="step", y="loss", alpha=0.3, label="loss / step", figsize=(9, 3))
loss.assign(smooth=loss.loss.rolling(50, min_periods=1).mean()).plot(x="step", y="smooth", ax=ax, label="50-step mean")
plt.show()
print(open("checkpoints/lora/ravi_varma/training_summary.json").read())
!ls -la checkpoints/lora/ravi_varma checkpoints/lora/ravi_varma/checkpoint-*/pytorch_lora_weights.safetensors

for grid in sorted(glob.glob("outputs/logs/validation_images/step_*/grid.jpg")):
    print(grid); im = Image.open(grid); im.thumbnail((900, 900)); display(im)

## 10. Image generation — sample prompts

Renders the five-prompt suite in `configs/prompts/sample_prompts.yaml` with three seeds using the
final LoRA, and the same prompts/seeds with the **base model** for comparison.

In [ ]:
!python scripts/generate.py --prompt-file configs/prompts/sample_prompts.yaml --seeds 42 43 44 --output-dir outputs/generated/lora
!python scripts/generate.py --prompt-file configs/prompts/sample_prompts.yaml --seeds 42 43 44 --no-lora --output-dir outputs/generated/baseline

**Checkpoint selection** (optional): render the same prompts from earlier checkpoints, and a
LoRA-strength sweep, to pick the checkpoint/scale that keeps the style without degrading faces and hands.

In [ ]:
for step in (750, 1000, 1250):
    !python scripts/generate.py --prompt-file configs/prompts/sample_prompts.yaml --seeds 42 --lora-path checkpoints/lora/ravi_varma/checkpoint-{step} --output-dir outputs/generated/ckpt_{step}
for scale in (0.6, 0.8):
    !python scripts/generate.py --prompt-file configs/prompts/sample_prompts.yaml --seeds 42 --lora-scale {scale} --output-dir outputs/generated/scale_{scale}

## 11. Result inspection and basic evaluation

In [ ]:
!python scripts/evaluate.py --input outputs/generated/lora --baseline outputs/generated/baseline --output-dir outputs/evaluations/lora_vs_base --device cuda
for sheet in ("contact_sheet.jpg", "contact_sheet_baseline.jpg"):
    print(sheet); display(Image.open(f"outputs/evaluations/lora_vs_base/{sheet}"))

## 12. Export

Downloads the final LoRA (weights + config + summary + loss history), the checkpoint LoRA files,
validation grids, generated images and the evaluation report. Unzip it in the local project root:
`unzip -o ravi_varma_results.zip`.

In [ ]:
import zipfile, glob
with zipfile.ZipFile("ravi_varma_results.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    patterns = [
        "checkpoints/lora/ravi_varma/*.safetensors", "checkpoints/lora/ravi_varma/*.yaml",
        "checkpoints/lora/ravi_varma/*.json", "checkpoints/lora/ravi_varma/*.csv",
        "checkpoints/lora/ravi_varma/checkpoint-*/pytorch_lora_weights.safetensors",
        "outputs/logs/validation_images/**/*", "outputs/logs/train_loss.csv",
        "outputs/generated/**/*", "outputs/evaluations/**/*",
    ]
    for pattern in patterns:
        for path in glob.glob(pattern, recursive=True):
            if os.path.isfile(path):
                zf.write(path)
print(f"{os.path.getsize('ravi_varma_results.zip') / 1e6:.0f} MB")
from google.colab import files
files.download("ravi_varma_results.zip")

## What this notebook does *not* cover (remaining milestones)

- Milestone 3: LLM-based prompt expansion / semantic scene parsing
- Milestone 4: ControlNet pose conditioning, full evaluation framework, web dashboard